In [1]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [2]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ==========================================
# 1. Configuration & Setup
# ==========================================
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# Set device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. Data Preparation (IMDB Dataset)
# ==========================================
print("Loading IMDB Dataset...")
# Load the actual CSV file downloaded from Kaggle
df = pd.read_csv(f'{path}/IMDB Dataset.csv')

# Randomly sample 2000 rows to ensure reasonable training time on CPU
df = df.sample(n=2000, random_state=42)

# Map string labels to numeric labels required by BERT
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Rename columns to match what our Custom Dataset expects
df = df.rename(columns={'review': 'text', 'sentiment': 'label'})

# Split into training (80%) and validation (20%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)
print(f"Training on {len(train_texts)} rows, validating on {len(val_texts)} rows.")

# ==========================================
# 3. Custom PyTorch Dataset
# ==========================================
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # Tokenize the text and pad/truncate to max length
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize Tokenizer and Create DataLoaders
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ==========================================
# 4. Model Initialization
# ==========================================
print("Initializing BERT Model...")
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# 5. Training Loop
# ==========================================
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

    return total_loss / len(data_loader)

# ==========================================
# 6. Evaluation Loop
# ==========================================
def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)

            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())

    return accuracy_score(actual_labels, predictions)

# ==========================================
# 7. Execution
# ==========================================
print("Starting training...")
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_acc = eval_model(model, val_loader, device)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

# Save the fine-tuned model (Optional but highly recommended for the assignment)
print("Saving model to disk...")
model.save_pretrained('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')
print("Training complete and model saved!")

Using device: cuda
Loading IMDB Dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'IMDB Dataset.csv'

In [3]:
!pip install transformers kagglehub tqdm

In [4]:
import torch
import pandas as pd
import kagglehub
import os
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ==========================================
# 1. Configuration & Setup
# ==========================================
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# Set device (Colab will automatically grab the GPU if you selected T4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. Data Preparation (KaggleHub Integration)
# ==========================================
print("Downloading IMDB Dataset from Kaggle...")
# Automatically download and locate the hidden dataset folder
dataset_folder_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
csv_file_path = os.path.join(dataset_folder_path, "IMDB Dataset.csv")

print(f"Loading CSV from: {csv_file_path}")
df = pd.read_csv(csv_file_path)

# Optional: If you want to test the pipeline quickly before doing a full 50,000 row run,
# uncomment the line below to grab a smaller sample.
# df = df.sample(n=5000, random_state=42)

# Map string labels to numeric labels required by BERT
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Rename columns to match what our Custom Dataset expects
df = df.rename(columns={'review': 'text', 'sentiment': 'label'})

# Split into training (80%) and validation (20%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)
print(f"Training on {len(train_texts)} rows, validating on {len(val_texts)} rows.")

# ==========================================
# 3. Custom PyTorch Dataset
# ==========================================
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize Tokenizer and Create DataLoaders
print("Initializing Tokenizer...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ==========================================
# 4. Model Initialization
# ==========================================
print("Initializing BERT Model...")
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# 5. Training Loop
# ==========================================
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0

    # Wrapped in tqdm for a clean progress bar
    for batch in tqdm(data_loader, desc="Training Epoch"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(data_loader)

# ==========================================
# 6. Evaluation Loop
# ==========================================
def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)

            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())

    return accuracy_score(actual_labels, predictions)

# ==========================================
# 7. Execution
# ==========================================
print("Starting training...")
for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch + 1} / {EPOCHS} ---")
    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_acc = eval_model(model, val_loader, device)
    print(f"Train Loss: {train_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

print("\nSaving model to Colab disk...")
model.save_pretrained('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')
print("Training complete and model saved!")

Using device: cuda
Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Loading CSV from: /kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv
Training on 40000 rows, validating on 10000 rows.
Initializing Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Train Loss: 0.1807 | Validation Accuracy: 0.8942

--- Epoch 3 / 3 ---


Evaluating: 100%|██████████| 625/625 [01:28<00:00,  7.03it/s]

Train Loss: 0.0930 | Validation Accuracy: 0.8920

Saving model to Colab disk...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!


# New Section